<a href="https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



For Lane 2, the unit of analysis is **one content item for one client**. The source performance table is daily, with one row per client × content item × report date, so the daily records are aggregated to the content-item level for the analysis.

For this contract, I will use **March 2026** as the development window. The March records provide the observed performance signals used to construct the content-level feature frame. I will not use the final-month `_sample` for developing label logic because it represents the final June 2026 outcome period.

The goal is to produce a page-level priority score that helps a content reviewer decide which pages should be reviewed first.


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# ML-04 Section 1
# Connect to the FlyRank warehouse through DuckDB.
# HF_TOKEN must be stored securely in Colab Secrets.

%pip -q install duckdb huggingface_hub

import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Add your READ Hugging Face token "
        "to Colab Secrets as HF_TOKEN."
    )

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

PERFORMANCE = (
    "read_parquet("
    "'hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet'"
    ")"
)

print("Warehouse connection ready.")
print("Development window: March 2026")

Warehouse connection ready.
Development window: March 2026


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



**Features:** March 2026 performance and content signals that are available at the decision moment, such as GSC impressions, clicks, average search position, GA4 sessions, engagement signals, content age, and days since last update.

**Label / proxy:** The future content outcome used to evaluate the ranking. It must be calculated from a later time window and must not be included among the features.

**Context:** `client_hash_id`, `content_hash_id`, and `report_date`. These fields identify, group, join, or order observations but should not be used as predictive features.

**Excluded:** Label-derived fields, future-window information, and existing product decision fields such as health or priority flags. These fields are excluded because they either reveal the outcome or reproduce an existing decision rule rather than providing independent evidence.

Missing values will be treated as meaningful where appropriate. In particular, an unavailable GA4 measurement must not automatically be interpreted as zero engagement. The warehouse documentation states that rows before a client's GA4 availability date can contain zero-filled GA4 values with `ga4_data_available = FALSE`.


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# ML-04 Section 2
# Define the fields we consider safe, label-related, contextual, or excluded.

feature_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "content_age_days",
    "days_since_last_update"
]

label_fields = [
    "future_gsc_impressions",
    "future_gsc_clicks",
    "future_outcome"
]

context_fields = [
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

excluded_fields = [
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier"
]

print("FEATURES:")
for x in feature_fields:
    print(" -", x)

print("\nLABEL / PROXY:")
for x in label_fields:
    print(" -", x)

print("\nCONTEXT:")
for x in context_fields:
    print(" -", x)

print("\nEXCLUDED:")
for x in excluded_fields:
    print(" -", x)

FEATURES:
 - gsc_impressions
 - gsc_clicks
 - gsc_avg_position
 - ga4_sessions
 - ga4_engaged_sessions
 - content_age_days
 - days_since_last_update

LABEL / PROXY:
 - future_gsc_impressions
 - future_gsc_clicks
 - future_outcome

CONTEXT:
 - client_hash_id
 - content_hash_id
 - report_date

EXCLUDED:
 - trend_direction
 - trend_pct
 - health_score
 - priority_score
 - action_type
 - refresh_tier


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


I will verify the contract against the March 2026 warehouse partition rather than relying only on documentation. The checks confirm that the source has the expected daily grain, show the number of records and date range in the selected month, and measure how many records have usable search and analytics availability.

The availability check uses the explicit boolean availability fields rather than interpreting zero-filled measurements as real zero activity.


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Query 1 — Verify the grain
# Expected grain: one row per client × content × report date.

grain_check = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM {PERFORMANCE}
    GROUP BY
        client_hash_id,
        content_hash_id,
        report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate rows at the expected grain:")
display(grain_check)

if len(grain_check) == 0:
    print("PASS: one row per client × content × report_date.")
else:
    print("CHECK REQUIRED: duplicate grain detected.")

# Query 2 — Verify row count and date span for March 2026

count_window = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {PERFORMANCE}
""").df()

display(count_window)

# Query 3 — Verify data availability.
# IS TRUE is intentional: FALSE means the source was not available,
# not that the metric was genuinely zero.

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows

    FROM {PERFORMANCE}
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows at the expected grain:


,client_hash_id,content_hash_id,report_date,row_count


PASS: one row per client × content × report_date.


,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347



### Five initial features

For the Lane 2 ranking task, I will start with five measurable features from the March 2026 development window:

1. **GSC impressions** — available before the decision because they summarize observed search exposure during the feature window.
2. **GSC clicks** — available before the decision because they summarize observed search traffic during the feature window.
3. **Average search position** — available before the decision because it is measured during the feature window. Values of 0 are treated as missing because 0 means no position data.
4. **GA4 sessions** — available before the decision when GA4 data is available for the observation.
5. **Observed days** — available before the decision because it counts the number of days with observed performance data for the content item during March.

These five features provide search exposure, search response, on-site activity, and data-coverage information without using future outcomes or label-derived trend fields.


In [24]:
# Build a small five-feature frame from March 2026.
# One row = one client × content item for the March development window.

# Check that the March 2026 performance partition is accessible

test = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {PERFORMANCE}
""").df()

display(test)


# Five initial features for the Lane 2 ranking task.
# One row = one client × content item in March 2026.

five_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- Feature 1: search impressions
        SUM(gsc_impressions) AS impressions_mar,

        -- Feature 2: search clicks
        SUM(gsc_clicks) AS clicks_mar,

        -- Feature 3: average search position
        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
                ELSE NULL
            END
        ) AS avg_position_mar,

        -- Feature 4: GA4 sessions
        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN ga4_sessions
                ELSE NULL
            END
        ) AS sessions_mar,

        -- Feature 5: number of observed days
        COUNT(DISTINCT report_date) AS observed_days_mar

    FROM {PERFORMANCE}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print(f"Feature rows: {len(five_features):,}")
print("Feature columns:")
print(list(five_features.columns))

display(five_features.head())

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 331,437
Feature columns:
['client_hash_id', 'content_hash_id', 'impressions_mar', 'clicks_mar', 'avg_position_mar', 'sessions_mar', 'observed_days_mar']


,client_hash_id,content_hash_id,impressions_mar,clicks_mar,avg_position_mar,sessions_mar,observed_days_mar
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,0.0,5.331238,NaN,31
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,46.0,1.0,5.942308,NaN,31
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5.908100,NaN,31
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,6.419872,NaN,31
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,0.0,6.969536,NaN,31


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


This data can support observed and directional content-performance analysis, but it cannot prove that a content change caused a performance change. The warehouse has uneven history across clients, so the amount of usable historical data is not identical for every client.

GA4 values also require an availability check because unavailable periods may contain zero-filled values. Therefore, a zero metric cannot automatically be interpreted as zero real activity.

The March 2026 development window is useful for building and testing the data contract, but it should not be treated as a final unbiased estimate of future performance. The final month is reserved as a sealed outcome window, and any future model should use non-overlapping feature and outcome windows.

The resulting analysis is therefore decision-support based on observed data, not causal proof or a guarantee of what will happen after a page is changed.


In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check how many rows have unavailable GSC or GA4 measurements.

availability_limits = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS NOT TRUE
        ) AS gsc_unavailable_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS NOT TRUE
        ) AS ga4_unavailable_rows

    FROM {PERFORMANCE}
""").df()

display(availability_limits)

,total_rows,gsc_unavailable_rows,ga4_unavailable_rows
0,9841378,6230317,9427412


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.